# ⚡ VibraDistill-Edge: ADXL345 Native Training & Dual-T4 Kaggle Pipeline
### Complete Hardware-Software Co-Design for Analog Devices ADXL345 (Fs = 3,200 Hz)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code)

> **Hardware Target:**
> - **Physical Accelerometer:** Analog Devices ADXL345 via SPI @ 5.0 MHz (ODR = 3,200 Hz, Nyquist = 1,600 Hz, Full Res $\pm 16g$)
> - **Physical Actuator:** DC Motor 130 + Optical Encoder 30 PPR (900 - 2000 RPM, $f_r = 15 - 33.3$ Hz)
> - **FPGA NPU:** Sipeed Tang Primer 20K (Gowin GW2A-LV18 @ 100 MHz, 8,677 INT8 weights, 6 BSRAMs, Zero DDR3)
> - **MCU Supervisor:** Sonix SN32F407 (ARM Cortex-M0 @ 60 MHz, HopACP Q16.16 Conformal Predictor + IEC 61508 SIL-2 Relay Trip)
> 
> **DSP Innovation:** 32-Tap Causal Minimum-Phase FIR ($[280, 1400]$ Hz) reducing group delay from 15.5 to **2.76 samples** (zero Gibbs ringing).
> 
> **Kaggle Configuration:**
> - **Accelerator:** `GPU T4 x2` (30 GB total VRAM) or `GPU T4 x1`
> - **Internet:** `On`
> - **Persistence:** `Variables and Files`


## 1. Verify GPU Allocation & CUDA Environment
Inspects available Tesla T4 GPUs (`cuda:0`, `cuda:1`).


In [ ]:
!nvidia-smi
import torch

print(f"PyTorch Version: {torch.__version__}")
num_gpus = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | GPU Count: {num_gpus}")

for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU [{i}]: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute Cap: {props.major}.{props.minor}")

dev = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Primary Compute Device: {dev}")


## 2. Setup Codebase & Synchronize from GitHub (Branch `tai`)
Clones or pulls the latest repository code from `zok213/porygon` branch `tai`.


In [ ]:
import os, sys, shutil, subprocess

WORKSPACE_DIR = "/kaggle/working/VIbraDistill" if os.path.exists("/kaggle") else os.getcwd()

if os.path.exists("/kaggle"):
    if not os.path.exists(WORKSPACE_DIR):
        print("[1/3] Cloning repository branch tai...")
        subprocess.run(["git", "clone", "-b", "tai", "https://github.com/zok213/porygon.git", WORKSPACE_DIR], check=True)
    else:
        print("[1/3] Updating repository branch tai...")
        subprocess.run(["git", "-C", WORKSPACE_DIR, "fetch", "origin", "tai"], check=True)
        subprocess.run(["git", "-C", WORKSPACE_DIR, "checkout", "tai"], check=True)
        subprocess.run(["git", "-C", WORKSPACE_DIR, "pull", "origin", "tai"], check=True)
    os.chdir(WORKSPACE_DIR)

# Add workspace root to sys.path
if WORKSPACE_DIR not in sys.path:
    sys.path.insert(0, WORKSPACE_DIR)

# Map or download CWRU dataset
os.makedirs("data", exist_ok=True)
cwru_path = "data/CWRU_Dataset/Train_7mil"
if not os.path.exists(cwru_path) or len(os.listdir(cwru_path)) == 0:
    print("[2/3] Downloading CWRU dataset...")
    subprocess.run([sys.executable, "scripts/download_all_benchmarks.py", "--dataset", "cwru"], check=True)
else:
    print(f"[2/3] CWRU Dataset verified ready ({len(os.listdir(cwru_path))} files in Train_7mil).")

print(f"[3/3] Active Working Directory: {os.getcwd()}")


## 3. ADXL345 Frequency Response & Bit-True 32-Tap Minimum-Phase FIR Verification
Analyzes why ADXL345 ($F_s = 3,200$ Hz) completely satisfies Nyquist requirements for Motor 130 and verifies the 32-tap Minimum-Phase FIR filter response.


In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
from src.dsp.causal_fir import RTL_ADXL345_HEX_COEFFS, RTL_ADXL345_FIR_COEFFS_INT16, RTL_ADXL345_FIR_COEFFS_Q15

fs = 3200.0
nyquist = fs / 2.0
q15 = RTL_ADXL345_FIR_COEFFS_Q15

# Compute frequency response and group delay
w, h = signal.freqz(q15, worN=2048)
f = w * fs / (2 * np.pi)
mag_db = 20 * np.log10(np.abs(h) + 1e-12)
_, gd = signal.group_delay((q15, 1.0), w=2048)

p_idx = np.where(mag_db >= -3.0)[0]
print("=" * 75)
print("  ADXL345 32-TAP MINIMUM-PHASE FIR SPECIFICATIONS (Fs = 3,200 Hz)")
print("=" * 75)
print(f"  Passband (-3 dB): [{f[p_idx[0]]:.1f} Hz, {f[p_idx[-1]]:.1f} Hz]")
print(f"  Mean Group Delay in Passband: {np.mean(gd[p_idx]):.2f} samples (vs 15.5 of linear-phase)")
print(f"  Stopband Attenuation at 60 Hz: {mag_db[np.argmin(np.abs(f - 60))]:.2f} dB (rejects shaft unbalance/hum)")
print(f"  Stopband Attenuation at 1550 Hz: {mag_db[np.argmin(np.abs(f - 1550))]:.2f} dB (anti-aliasing near Nyquist)")
print("=" * 75)


## 4. Run End-to-End ADXL345 Training & Silicon Export Pipeline
Executes all 6 stages:
1. **Polyphase Decimation**: CWRU $12.0\text{ kHz} \to 3.2\text{ kHz}$.
2. **Teacher ResNet-18 Training**: 25 epochs.
3. **Decoupled Knowledge Distillation (DKD)**: Into `VibraDistillMicro` (8,677 parameters, 30 epochs).
4. **Quantization-Aware Training (QAT)**: 15 epochs with Straight-Through Estimators.
5. **Bit-True Gowin NPU 12-Way Hardware Emulation**.
6. **Export**: 6 BSRAM `.hex`/`.mi` banks, unified `npu_weights_adxl345_int8.mem`, Sonix `vibradistill_adxl345_weights.h`, and `fir_adxl345_32.v`.


In [ ]:
dev = "cuda:0" if torch.cuda.is_available() else "cpu"

!python scripts/train_adxl345_native.py \
    --data_dir data/CWRU_Dataset \
    --cache_dir data/cache \
    --teacher_epochs 40 \
    --student_epochs 50 \
    --qat_epochs 25 \
    --batch_size 64 \
    --device {dev}


## 5. Verify Exported Artifacts & Bit-True Parity
Confirms exported FPGA BRAM files, MCU headers, and audit metrics.


In [ ]:
import json

report_file = "experiments/adxl345_training_report.json"
if os.path.exists(report_file):
    with open(report_file, "r") as f:
        r = json.load(f)
    print("=" * 75)
    print("  AUDIT SUMMARY: VIBRADISTILL-EDGE (ADXL345 NATIVE)")
    print("=" * 75)
    for k, v in r.items():
        print(f"  - {k:30s}: {v}")
    print("=" * 75)
else:
    print("Audit report not found!")


## 6. Package Output Artifacts for 1-Click Download
Creates `vibradistill_adxl345_artifacts.zip` containing:
- Trained checkpoints (`checkpoints/`)
- FPGA BSRAM memory files (`embedded/fpga_gowin/roms/`)
- Sonix MCU headers (`embedded/mcu_sonix/app/`)
- Verilog RTL module (`embedded/fpga_gowin/src/fir_adxl345_32.v`)
- Audit JSON (`experiments/adxl345_training_report.json`)


In [ ]:
import zipfile

out_zip = "/kaggle/working/vibradistill_adxl345_artifacts.zip" if os.path.exists("/kaggle") else "vibradistill_adxl345_artifacts.zip"
print(f"Creating download archive: {out_zip}...")

targets = [
    "checkpoints/teacher_adxl345",
    "checkpoints/student_dkd_adxl345",
    "checkpoints/student_qat_adxl345",
    "embedded/fpga_gowin/roms/adxl345",
    "embedded/fpga_gowin/roms/npu_weights_adxl345_int8.mem",
    "embedded/fpga_gowin/src/fir_adxl345_32.v",
    "embedded/mcu_sonix/app/vibradistill_adxl345_weights.h",
    "experiments/adxl345_training_report.json"
]

with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for target in targets:
        if os.path.exists(target):
            if os.path.isdir(target):
                for root, _, files in os.walk(target):
                    for f in files:
                        fp = os.path.join(root, f)
                        z.write(fp, os.path.relpath(fp, '.'))
            else:
                z.write(target, os.path.relpath(target, '.'))

size_mb = os.path.getsize(out_zip) / (1024 * 1024)
print(f"Successfully packaged {size_mb:.2f} MB into {out_zip}!")
print("Download this file from the Kaggle Output tab on the right sidebar.")
